In [90]:
import re
from collections import Counter
import math

# Pipeline de PLN con Naive Bayes

### 1. Representación
- Tokenización
- Extracción de N-Gramas

### 2. Robustez
- Suavizado de Laplace

### 3. Categorización
- Implementacion del modelo Naive Bayes

### 4. Evaluación
- Aplicar un métrica como:
    - Matriz de Confusión
    - F1
    - Accuracy
    - Recall

## 1. Preprocesamiento

In [91]:
STOPWORDS = {
    "el",
    "la",
    "los",
    "las",
    "un",
    "una",
    "unos",
    "unas",
    "de",
    "en",
    "y",
    "a",
    "que",
    "se",
    "no",
    "es",
    "son",
    "por",
    "con",
    "para",
    "del",
    "al",
    "lo",
    "le",
    "su",
}


def normalizar(texto):
    texto = texto.lower()
    texto = re.sub(r"[^a-záéíóúüñ\s]", "", texto)  # solo letras
    texto = re.sub(r"\s+", " ", texto).strip()
    return texto


def tokenizar(texto):
    return texto.split()


def eliminar_stopwords(tokens):
    return [t for t in tokens if t not in STOPWORDS]


def stemming_simple(token):
    """Stemmer muy básico para español"""
    sufijos = [
        "iendo",
        "ando",
        "iones",
        "cion",
        "mente",
        "ados",
        "adas",
        "idos",
        "idas",
        "es",
        "s",
    ]
    for suf in sufijos:
        if token.endswith(suf) and len(token) - len(suf) >= 3:
            return token[: -len(suf)]
    return token


def preprocesar(texto):
    texto = normalizar(texto)
    tokens = tokenizar(texto)
    tokens = eliminar_stopwords(tokens)
    stems = [stemming_simple(t) for t in tokens]

    return stems

## 3. Modelo Naive Bayes
- Naive Bayes aplicado para identificar sentimientos en un texto
### 3.1 Probabilidad
- Primero, es necesario calcular la probabilidad de que un documento o texto analizado pertenezca a una de las clases con las que fue entrenado el modelo. Para ello, se utiliza la siguiente fórmula:
$$P(Clase|Documento) \propto P(Clase) \prod P(w_i| Clase)$$
- P(Clase) se conoce como **Prior**, y es la proporcion de documentos de cada clase en el corupus de entrenamiento.
    - Por ejemplo, si se tienen 70 reseñas positivas y 30 reseñas negativas, entonces:
        - $P(\text{Positivas})=0.7$
        - $P(\text{Negativas})=0.3$

In [92]:
class NaiveBayesSentimiento:
    def __init__(self):

        self.log_priors = {}  # log P(clase)

        self.log_likelihoods = {}  # log P(w | clase)

        self.vocab = set()

        self.clases = []

    def entrenar(self, documentos, etiquetas):
        """documentos: list[str], etiquetas: list[str]"""

        self.clases = list(set(etiquetas))

        N = len(documentos)

        for clase in self.clases:
            # Documentos de esta clase

            docs_c = [documentos[i] for i, e in enumerate(etiquetas) if e == clase]

            self.log_priors[clase] = math.log(len(docs_c) / N)

            # Concatenar todos los tokens de esta clase

            tokens_c = []

            for doc in docs_c:
                tokens_c.extend(preprocesar(doc))

            self.vocab.update(tokens_c)

            conteo = Counter(tokens_c)

            total = sum(conteo.values())

            self.log_likelihoods[clase] = {
                w: math.log((conteo[w] + 1) / (total + len(self.vocab))) for w in conteo
            }

            # Guardar total para palabras desconocidas

            self.log_likelihoods[clase]["__total__"] = total

    def clasificar(self, texto):

        tokens = preprocesar(texto)

        scores = {}

        for clase in self.clases:
            score = self.log_priors[clase]

            V = len(self.vocab)

            total = self.log_likelihoods[clase]["__total__"]

            for token in tokens:
                if token in self.log_likelihoods[clase]:
                    score += self.log_likelihoods[clase][token]

                else:  # Laplace para palabras no vistas
                    score += math.log(1 / (total + V))

            scores[clase] = score

        return max(scores, key=scores.get), scores

## 3. Entrenamiento del Modelo

In [93]:
datos = [
    "la pelicula es buena me encanto",
    "excelente historia muy emocionante",
    "me encanto el final fue hermoso",
    "la pelicula es terrible muy aburrida",
    "mala historia no me gusto",
    "horrible fue muy lenta y aburrida",
]

labels = ["POS", "POS", "POS", "NEG", "NEG", "NEG"]

modelo = NaiveBayesSentimiento()

modelo.entrenar(datos, labels)

## 4. Testing del Modelo

In [94]:
pred, scores = modelo.clasificar("excelente historia")

print(f"Predicción: {pred}")

print(f"Scores: {scores}")

Predicción: POS
Scores: {'NEG': -6.579251212010101, 'POS': -6.1092475827643655}
